<a href="https://colab.research.google.com/github/Arhammanj/Arhammanj/blob/main/MicroGridEnv.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
!pip install gym numpy matplotlib


In [4]:
import gym
import numpy as np
from gym import spaces

ACTIONS = [i for i in range(-80, 100, 20)]  # battery charge/discharge actions

class Battery:
    def __init__(self, capacity=400, max_soc=0.8, min_soc=0.2, efficiency=1, degradation=0.001):
        self.capacity = capacity
        self.max_soc = max_soc
        self.min_soc = min_soc
        self.efficiency = efficiency
        self.degradation = degradation
        self.current_capacity = np.random.uniform(min_soc, max_soc)

    def step(self, action):
        energy = action
        updated_capacity = max(self.min_soc, min(self.max_soc,
                                (self.current_capacity*self.capacity + energy)/self.capacity))
        self.energy_change = (updated_capacity - self.current_capacity) * self.capacity
        self.current_capacity = updated_capacity

    def cost(self, energy):
        return (energy**2) * self.degradation

    @property
    def SOC(self):
        return self.current_capacity

    def reset(self):
        self.current_capacity = np.random.uniform(self.min_soc, self.max_soc)
        return self.current_capacity


class Grid:
    def __init__(self, prices=None):
        if prices is None:
            prices = np.linspace(0.05, 0.15, 24)  # simple price curve
        self.sell_prices = prices
        self.time = 0

    def set_time(self, t):
        self.time = t

    def cost(self, p_ex):
        return self.sell_prices[self.time % 24] * p_ex


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [5]:
class MicroGridEnvA(gym.Env):
    """Option A: Normalized Economic + Stability"""
    def __init__(self):
        super().__init__()
        self.iterations = 24
        self.time_step = 0
        self.grid = Grid()
        self.battery = Battery()
        self.action_space = spaces.Discrete(len(ACTIONS))
        self.observation_space = spaces.Box(low=0, high=1, shape=(3,), dtype=np.float32)

    def step(self, action):
        action = ACTIONS[action]
        self.grid.set_time(self.time_step)
        self.battery.step(action)

        total_output = self.battery.energy_change
        buy_cost, sell_benefit = 0, 0
        if total_output >= 0:
            buy_cost = self.grid.cost(total_output)
        else:
            sell_benefit = self.grid.cost(abs(total_output))

        battery_cost = self.battery.cost(self.battery.energy_change)
        dg_cost = 0
        excess_penalty, deficient_penalty, soc_penalty = 0, 0, 0

        reward = (sell_benefit - (battery_cost + dg_cost + buy_cost)) / (
                    1 + excess_penalty + deficient_penalty + soc_penalty
                 )

        self.time_step += 1
        state = np.array([self.time_step/self.iterations, self.battery.SOC, self.grid.sell_prices[self.time_step % 24]])
        done = self.time_step == self.iterations
        return state, reward, done, {}

    def reset(self):
        self.time_step = 0
        self.battery.reset()
        return np.array([0, self.battery.SOC, self.grid.sell_prices[0]])


class MicroGridEnvB(MicroGridEnvA):
    """Option B: Weighted Multi-Objective"""
    def step(self, action):
        action = ACTIONS[action]
        self.grid.set_time(self.time_step)
        self.battery.step(action)

        total_output = self.battery.energy_change
        buy_cost, sell_benefit = 0, 0
        if total_output >= 0:
            buy_cost = self.grid.cost(total_output)
        else:
            sell_benefit = self.grid.cost(abs(total_output))

        battery_cost = self.battery.cost(self.battery.energy_change)
        dg_cost = 0
        excess_penalty, deficient_penalty, soc_penalty = 0, 0, 0

        reward = -0.5*(battery_cost + dg_cost + buy_cost) \
                 + 1.0*sell_benefit \
                 - 0.7*(excess_penalty + deficient_penalty) \
                 - 0.3*soc_penalty

        self.time_step += 1
        state = np.array([self.time_step/self.iterations, self.battery.SOC, self.grid.sell_prices[self.time_step % 24]])
        done = self.time_step == self.iterations
        return state, reward, done, {}


class MicroGridEnvC(MicroGridEnvA):
    """Option C: Long-Term Efficiency"""
    def step(self, action):
        action = ACTIONS[action]
        self.grid.set_time(self.time_step)
        self.battery.step(action)

        total_output = self.battery.energy_change
        buy_cost, sell_benefit = 0, 0
        if total_output >= 0:
            buy_cost = self.grid.cost(total_output)
        else:
            sell_benefit = self.grid.cost(abs(total_output))

        battery_cost = self.battery.cost(self.battery.energy_change)
        excess_penalty, deficient_penalty = 0, 0

        reward = (sell_benefit - buy_cost) \
                 - 0.01*battery_cost \
                 - (excess_penalty + deficient_penalty)

        self.time_step += 1
        state = np.array([self.time_step/self.iterations, self.battery.SOC, self.grid.sell_prices[self.time_step % 24]])
        done = self.time_step == self.iterations
        return state, reward, done, {}


In [7]:
class MicroGridEnvBase(gym.Env):
    """Baseline Reward: sell_benefit - buy_cost - 0.01*battery_cost"""
    def __init__(self):
        super().__init__()
        self.iterations = 24
        self.time_step = 0
        self.grid = Grid()
        self.battery = Battery()
        self.action_space = spaces.Discrete(len(ACTIONS))
        self.observation_space = spaces.Box(low=0, high=1, shape=(3,), dtype=np.float32)

    def step(self, action):
        action = ACTIONS[action]
        self.grid.set_time(self.time_step)
        self.battery.step(action)

        total_output = self.battery.energy_change
        buy_cost, sell_benefit = 0, 0
        if total_output >= 0:
            buy_cost = self.grid.cost(total_output)
        else:
            sell_benefit = self.grid.cost(abs(total_output))

        battery_cost = self.battery.cost(self.battery.energy_change)

        # Baseline reward
        reward = sell_benefit - buy_cost - 0.01 * battery_cost

        self.time_step += 1
        state = np.array([self.time_step/self.iterations, self.battery.SOC, self.grid.sell_prices[self.time_step % 24]])
        done = self.time_step == self.iterations
        return state, reward, done, {}

    def reset(self):
        self.time_step = 0
        self.battery.reset()
        return np.array([0, self.battery.SOC, self.grid.sell_prices[0]])


In [9]:
# --- Training Loop ---
def train(env, episodes=50):
    rewards = []
    for ep in range(episodes):
        state = env.reset()
        total_reward = 0
        done = False
        while not done:
            action = env.action_space.sample()  # random policy for demo
            state, reward, done, _ = env.step(action)
            total_reward += reward
        rewards.append(total_reward)
    return rewards

In [10]:
envBase = MicroGridEnvBase()
envA = MicroGridEnvA()
envB = MicroGridEnvB()
envC = MicroGridEnvC()

for env, name in [(envBase, "Baseline"), (envA, "Option A"), (envB, "Option B"), (envC, "Option C")]:
    state = env.reset()
    total_reward = 0
    for _ in range(24):
        action = env.action_space.sample()
        state, reward, done, info = env.step(action)
        total_reward += reward
        if done:
            break
    print(f"{name} finished one episode with total reward: {total_reward:.4f}")


Baseline finished one episode with total reward: -5.5870
Option A finished one episode with total reward: -38.4739
Option B finished one episode with total reward: -13.5670
Option C finished one episode with total reward: 21.8580
